# 👕 Outfit Anyone Virtual Try-On (ComfyUI Colab)
**Outfit Anyone** is Alibaba's ultra-high-quality virtual try-on model capable of handling any person and any outfit.

--- 
Developed by **nano**

In [ ]:
#@title 1. Install Outfit Anyone Nodes
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install xformers -r requirements.txt

%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!git clone https://github.com/HumanAIGC/OutfitAnyone.git || echo "Using Wrapper instead"
!git clone https://github.com/chflame163/ComfyUI_CatVTON_Wrapper.git # OutfitAnyone is often combined with CatVTON workflow

%cd /content/ComfyUI/custom_nodes/ComfyUI_CatVTON_Wrapper
!pip install -r requirements.txt

In [ ]:
#@title 2. Download Outfit Anyone Weights
%cd /content/ComfyUI/models/checkpoints
!apt-get -y install -qq aria2
# Loading core weights for Outfit Anyone style workflows
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/HumanAIGC/OutfitAnyone/resolve/main/outfit_anyone_weights.safetensors || echo "Check official repo for direct links"

In [ ]:
#@title 3. Launch
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess, threading, time, socket
def iframe_thread(port):
  while True:
      time.sleep(0.5)
      if socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect_ex(('127.0.0.1', port)) == 0: break
  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    if "trycloudflare.com " in line.decode(): print("URL:", line.decode()[line.decode().find("http"):])

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server